[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/deployment.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239303-lesson-8-deployment)

# 部署

## 回顾

我们构建了一个带有记忆的 Agent：

* `act` —— 让模型调用特定工具
* `observe` —— 将工具输出传回给模型
* `reason` —— 让模型根据工具输出推理下一步该做什么（例如，调用另一个工具或直接响应）
* `persist state` —— 使用内存 Checkpointer 支持带中断的长时间对话

## 目标

现在，我们将介绍如何将 Agent 实际部署到本地 Studio 和 `LangGraph Cloud`。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph_sdk langchain_core

## 核心概念

需要理解几个核心概念：

`LangGraph` ——
- Python 和 JavaScript 库
- 允许创建 Agent 工作流

`LangGraph API` ——
- 打包图代码
- 提供任务队列以管理异步操作
- 提供持久化以维护跨交互的状态

`LangSmith Deployment`（原 `LangGraph Cloud`）——
- LangGraph API 的托管服务
- 允许从 GitHub 仓库部署图
- 还为已部署的图提供监控和追踪功能
- 每个部署可通过唯一的 URL 访问

`LangSmith Studio`（原 `LangGraph Studio`）——
- LangGraph 应用的集成开发环境（IDE）
- 使用 API 作为后端，允许实时测试和探索图
- 可以在本地运行，也可以与云部署一起运行。参见下文。

`LangGraph SDK` ——
- 用于编程方式与 LangGraph 图交互的 Python 库
- 为本地或云端提供的图提供一致的接口
- 允许创建客户端、访问 Assistant、管理线程和执行运行

## 本地测试

## Studio

**⚠️ 注意**

自录制这些视频以来，我们已更新了 Studio，使其现在可以在本地运行并通过浏览器访问。这是运行 Studio 的首选方式，而不是使用视频中展示的桌面应用。它现在被称为 *LangSmith Studio* 而不是 *LangGraph Studio*。详细的设置说明可在课程开头的"Getting Setup"指南中找到。你可以[在此](https://docs.langchain.com/langsmith/studio)查看 Studio 的说明，以及[在此](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server)查看本地部署的具体细节。

要在本模块的 `/studio` 目录中启动本地开发服务器，请在终端中运行以下命令：

```
langgraph dev
```

你应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到上面显示的 **Studio UI** URL。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Unfortunately LangGraph Studio is currently not supported on Google Colab")

In [ ]:
from langgraph_sdk import get_client

In [ ]:
# 这是本地开发服务器的 URL
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# 搜索所有托管的图
assistants = await client.assistants.search()

In [ ]:
assistants[-3]

In [ ]:
# 创建一个用于追踪运行状态的线程
thread = await client.threads.create()

现在，我们可以使用 [client.runs.stream](https://docs.langchain.com/oss/python/langgraph/graph-api/#stream-and-astream) 来运行 Agent，需要提供：

* `thread_id`
* `graph_id`
* `input`
* `stream_mode`

我们将在后续模块中深入讨论流式传输。

目前，只需了解我们正在使用 `stream_mode="values"` [流式传输](https://docs.langchain.com/langsmith/streaming)图中每一步之后的完整状态值。

状态被捕获在 `chunk.data` 中。

In [ ]:
from langchain_core.messages import HumanMessage

# 输入
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# 流式传输
async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

## 使用 Cloud 进行测试

我们可以通过 LangSmith 部署到 Cloud，如[此处](https://docs.langchain.com/langsmith/deployment-quickstart#deploy-from-github-with-langgraph-cloud)所述。

### 在 GitHub 上创建一个新仓库

* 前往你的 GitHub 账户
* 点击右上角的 "+" 图标，选择 `"New repository"`
* 命名你的仓库（例如 `langchain-academy`）

### 将你的 GitHub 仓库添加为 Remote

* 回到你克隆 `langchain-academy` 的终端
* 将你新创建的 GitHub 仓库添加为 remote

```
git remote add origin https://github.com/your-username/your-repo-name.git
```
* 推送到该仓库
```
git push -u origin main
```

### 将 LangSmith 连接到你的 GitHub 仓库

* 前往 [LangSmith](https://smith.langchain.com/)
* 点击左侧 LangSmith 面板上的 `deployments` 选项卡
* 添加 `+ New Deployment`
* 然后，选择你刚刚为课程创建的 Github 仓库（例如 `langchain-academy`）
* 将 `LangGraph API config file` 指向其中一个 `studio` 目录
* 例如，对于 module-1 使用：`module-1/studio/langgraph.json`
* 设置你的 API 密钥（例如，你可以直接从 `module-1/studio/.env` 文件复制）

![Screenshot 2024-09-03 at 11.35.12 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fd61c93d48e5d0f47_deployment2.png)

### 使用你的部署

然后，我们可以通过几种不同的方式与部署进行交互：

* 通过 SDK，如前所述。
* 通过 [LangGraph Studio](https://docs.langchain.com/langsmith/deployment-quickstart#3-test-your-application-in-studio)。

![Screenshot 2024-08-23 at 10.59.36 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fa159a09a51d601de_deployment3.png)

要在 notebook 中使用 SDK，只需确保已设置 `LANGSMITH_API_KEY`！

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("LANGSMITH_API_KEY")

In [ ]:
# 将其替换为你已部署图的 URL
URL = "https://langchain-academy-8011c561878d50b1883f7ed11b32d720.default.us.langgraph.app"
client = get_client(url=URL)

# 搜索所有托管的图
assistants = await client.assistants.search()

In [ ]:
# 选择 agent
agent = assistants[0]

In [ ]:
agent

In [ ]:
from langchain_core.messages import HumanMessage

# 创建一个用于追踪运行状态的线程
thread = await client.threads.create()

# 输入
input = {"messages": [HumanMessage(content="Multiply 3 by 2.")]}

# 流式传输
async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])